# Anthropic Batch Workflow

This notebook covers:
1. Generate `requests.json`
2. Create a message batch
3. Poll until `processing_status == "ended"`
4. Download JSONL results
5. Parse them back into the standard artifacts


In [11]:
from pathlib import Path
import json
import subprocess
import time

from anthropic import Anthropic


In [12]:
INPUT_POOL = Path('output/05_all_concepts.txt')
BATCH_DIR = Path('output/anthropic_batch')
REQUESTS_PATH = BATCH_DIR / 'requests.json'
MANIFEST_PATH = BATCH_DIR / 'requests_manifest.json'
RESULTS_PATH = BATCH_DIR / 'results.jsonl'
BATCH_DIR.mkdir(parents=True, exist_ok=True)
api_key = 'REPLACE_WITH_API_KEY'
client = Anthropic(api_key=api_key)


In [20]:
subprocess.run([
    'python', 'prepare_anthropic_batch_requests.py',
    '--input', str(INPUT_POOL),
    '--output', str(BATCH_DIR),
], check=True)

REQUESTS_PATH, MANIFEST_PATH


17:40:52 [INFO] Saved batch payload   → output/anthropic_batch/requests.json
17:40:52 [INFO] Saved batch manifest  → output/anthropic_batch/requests_manifest.json
17:40:52 [INFO] Prepared 364 requests for 18176 concepts


(PosixPath('output/anthropic_batch/requests.json'),
 PosixPath('output/anthropic_batch/requests_manifest.json'))

In [21]:
with open(REQUESTS_PATH, 'r', encoding='utf-8') as f:
    payload = json.load(f)

len(payload['requests'])


364

In [22]:
message_batch = client.messages.batches.create(requests=payload['requests'])
message_batch.id, message_batch.processing_status


('msgbatch_01QSGdYhgRwAacspEiBopwoR', 'in_progress')

In [23]:
batch_id_path = BATCH_DIR / 'batch_id.txt'
batch_id_path.write_text(message_batch.id + '\n', encoding='utf-8')
batch_id_path


PosixPath('output/anthropic_batch/batch_id.txt')

In [24]:
batch_id = batch_id_path.read_text(encoding='utf-8').strip()
message_batch = client.messages.batches.retrieve(batch_id)
message_batch.id, message_batch.processing_status


('msgbatch_01QSGdYhgRwAacspEiBopwoR', 'in_progress')

In [25]:
while message_batch.processing_status != 'ended':
    print(message_batch.processing_status, message_batch.request_counts)
    time.sleep(30)
    message_batch = client.messages.batches.retrieve(batch_id)

message_batch.processing_status, message_batch.request_counts


in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)
in_progress MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=364, succeeded=0)


('ended',
 MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=364))

In [26]:
stream = client.messages.batches.results(batch_id)
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    for entry in stream:
        f.write(json.dumps(entry.to_dict(), ensure_ascii=False) + '\n')

RESULTS_PATH


PosixPath('output/anthropic_batch/results.jsonl')

In [27]:
subprocess.run([
    'python', 'parse_anthropic_batch_results.py',
    '--results', str(RESULTS_PATH),
    '--manifest', str(MANIFEST_PATH),
    '--output', 'output/anthropic',
], check=True)


CompletedProcess(args=['python', 'parse_anthropic_batch_results.py', '--results', 'output/anthropic_batch/results.jsonl', '--manifest', 'output/anthropic_batch/requests_manifest.json', '--output', 'output/anthropic'], returncode=0)